In [4]:
import pandas as pd
import numpy as np

# Cargar el dataset que acabamos de descargar de GitHub
df = pd.read_csv('../data/raw/SuperStoreOrders - SuperStoreOrders.csv')

# Crear la matriz de interacciones Cliente vs Producto
matriz_interaccion = df.pivot_table(
    index='customer_name', 
    columns='product_id', 
    values='quantity', 
    aggfunc='sum', 
    fill_value=0
)

print("¡Matriz de interacciones creada con éxito!")
print("Dimensiones (Clientes x Productos):", matriz_interaccion.shape)
matriz_interaccion.head()

¡Matriz de interacciones creada con éxito!
Dimensiones (Clientes x Productos): (795, 10292)


product_id,FUR-ADV-10000002,FUR-ADV-10000108,FUR-ADV-10000183,FUR-ADV-10000188,FUR-ADV-10000190,FUR-ADV-10000571,FUR-ADV-10000600,FUR-ADV-10000847,FUR-ADV-10001283,FUR-ADV-10001440,...,TEC-STA-10003330,TEC-STA-10003386,TEC-STA-10003447,TEC-STA-10003550,TEC-STA-10003925,TEC-STA-10004181,TEC-STA-10004536,TEC-STA-10004542,TEC-STA-10004834,TEC-STA-10004927
customer_name,,,,,,,,,,,,,,,,,,,,,
Aaron Bergman,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Aaron Hawkins,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Aaron Smayling,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Adam Bellavance,0,0,6,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Adam Hart,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
# Guardar la matriz de interacciones procesada en la carpeta de datos
matriz_interaccion.to_csv('../data/matriz_interaccion.csv')
print("¡Matriz guardada con éxito en la carpeta data!")

¡Matriz guardada con éxito en la carpeta data!


In [6]:
import pandas as pd
import numpy as np

def pipeline_ingenieria_caracteristicas(filepath):
    """Pipeline reproducible para la extracción de características del dataset SuperStore"""
    print("1. Cargando datos originales...")
    df = pd.read_csv(filepath)
    
    print("2. Generando Matriz de Interacciones (Cliente x Producto)...")
    matriz_interaccion = df.pivot_table(
        index='customer_name', 
        columns='product_id', 
        values='quantity', 
        aggfunc='sum', 
        fill_value=0
    )
    
    print("3. Extrayendo características agregadas por Cliente...")
    features_clientes = df.groupby('customer_name').agg(
        total_ordenes=('order_id', 'count'),
        gasto_total=('sales', 'sum'),
        ganancia_total=('profit', 'sum'),
        unidades_compradas=('quantity', 'sum')
    ).reset_index()
    
    print("4. Extrayendo características agregadas por Producto...")
    features_productos = df.groupby(['product_id', 'category', 'sub_category']).agg(
        veces_solicitado=('order_id', 'count'),
        unidades_totales_vendidas=('quantity', 'sum'),
        ingresos_totales=('sales', 'sum')
    ).reset_index()
    
    # Exportar resultados estructurados
    matriz_interaccion.to_csv('../data/matriz_interaccion.csv')
    features_clientes.to_csv('../data/features_clientes.csv', index=False)
    features_productos.to_csv('../data/features_productos.csv', index=False)
    
    print("¡Pipeline de características ejecutado y guardado con éxito!")
    return matriz_interaccion, features_clientes, features_productos

# Ejecutar el pipeline completo
matriz, cltes, prods = pipeline_ingenieria_caracteristicas('../data/raw/SuperStoreOrders - SuperStoreOrders.csv')

1. Cargando datos originales...
2. Generando Matriz de Interacciones (Cliente x Producto)...
3. Extrayendo características agregadas por Cliente...
4. Extrayendo características agregadas por Producto...
¡Pipeline de características ejecutado y guardado con éxito!


### Justificación Técnica de la Ingeniería de Características
- **Matriz de Interacción (Cliente x Producto):** Convierte las transacciones históricas en un espacio vectorial estructurado. Aborda directamente el problema de *sparsity* (matrices dispersas en comercio electrónico) permitiendo calcular distancias de afinidad.
- **Características Agregadas:** Resumen el comportamiento monetario y de volumen del cliente (gasto total, frecuencia), permitiendo segmentar perfiles en futuros sprints.

In [7]:
cltes.head(10)

,customer_name,total_ordenes,gasto_total,ganancia_total,unidades_compradas
0,Aaron Bergman,89,"1349182438210547513219807662821,03914375512401...",4683.20800,301
1,Aaron Hawkins,56,"248102798524210553,8172,1214978118249235191,02...",2450.92904,231
2,Aaron Smayling,60,2649788489662519983301519232892161467040714084...,369.16180,211
3,Adam Bellavance,68,"1830501,038214191281882921142719221,5798009510...",4979.97690,262
4,Adam Hart,84,4905845488116374536768152985177963846587411681...,1902.03342,293
5,Adam Shillingsburg,71,"3350811569580971341631025310144433850481,02456...",1421.27412,238
6,Adrian Barton,77,"77301,10441,0671966125142797143204463356711462...",6417.28450,265
7,Adrian Hane,54,2592188521382328421843973182180126182317311211...,2081.37928,207
8,Adrian Shami,44,"63921871,1825272418060164419210813433672113341...",1564.49380,144
9,Aimee Bixby,55,"17848098605019595804425425153,4421,6774343152,...",2926.35430,208


In [8]:
prods.head(10)

,product_id,category,sub_category,veces_solicitado,unidades_totales_vendidas,ingresos_totales
0,FUR-ADV-10000002,Furniture,Furnishings,2,3,10653
1,FUR-ADV-10000108,Furniture,Furnishings,3,7,10020050
2,FUR-ADV-10000183,Furniture,Furnishings,8,31,5321210695538553318
3,FUR-ADV-10000188,Furniture,Furnishings,5,7,1550102525
4,FUR-ADV-10000190,Furniture,Furnishings,1,2,222
5,FUR-ADV-10000571,Furniture,Furnishings,8,20,110439110219110878110219
6,FUR-ADV-10000600,Furniture,Furnishings,2,4,103103
7,FUR-ADV-10000847,Furniture,Furnishings,5,11,26532615926
8,FUR-ADV-10001283,Furniture,Furnishings,2,3,11167
9,FUR-ADV-10001440,Furniture,Furnishings,7,22,27014180360454545


In [9]:
matriz.head(5)

product_id,FUR-ADV-10000002,FUR-ADV-10000108,FUR-ADV-10000183,FUR-ADV-10000188,FUR-ADV-10000190,FUR-ADV-10000571,FUR-ADV-10000600,FUR-ADV-10000847,FUR-ADV-10001283,FUR-ADV-10001440,...,TEC-STA-10003330,TEC-STA-10003386,TEC-STA-10003447,TEC-STA-10003550,TEC-STA-10003925,TEC-STA-10004181,TEC-STA-10004536,TEC-STA-10004542,TEC-STA-10004834,TEC-STA-10004927
customer_name,,,,,,,,,,,,,,,,,,,,,
Aaron Bergman,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Aaron Hawkins,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Aaron Smayling,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Adam Bellavance,0,0,6,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Adam Hart,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
